# J2: Single-task backbone bench (on 2024 val)
Bench DLinear/sn168 floors + TiDE + N-HiTSx. Pick the winner for the joint MTL model. TFT is documented-and-rejected (see the spec), not benchmarked, to save MPS compute.

In [ ]:
import sys; sys.path.insert(0, '../../src')
import os; os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import pandas as pd, numpy as np, torch
from data.loaders import load_split
from module_joint.config import JointConfig, select_device
LQ = pd.read_parquet('../../data/module_a/load_quantiles.parquet')
device = select_device(); print('device', device)


In [ ]:
from module_joint.train import train_one
from module_joint.baselines import SeasonalNaive168, dlinear_floor_cfg
from module_joint.evaluate import compare_price, compare_load, make_truth, leaderboard
train, val = load_split('train'), load_split('val')
val_ctx = pd.concat([train.tail(168), val])


In [ ]:
rows = []
# seasonal-naive floor
sn = SeasonalNaive168().fit(train).predict_quantiles(val_ctx, restrict_to=val.index)
for tgt, name in [('price','sn168'), ]:
    pr = sn['price']; tr = make_truth(val_ctx, pr.index, 'price')
    m = compare_price(pr, tr)['overall']; m['model']='sn168'; rows.append(m)
# DL backbones (raise epochs for real runs)
for bb in ['dlinear','tide','nhitsx']:
    cfg = JointConfig(backbone=bb, max_epochs=40)
    est,_ = train_one(cfg, train, val, LQ, device=device)
    pr = est.predict_quantiles(val_ctx, LQ, restrict_to=val.index)['price']
    tr = make_truth(val_ctx, pr.index, 'price')
    m = compare_price(pr, tr)['overall']; m['model']=bb; rows.append(m)
leaderboard(rows)
